## PDF Derivation

Given two measurements from a log normal distribution with the same mean and $CV=\theta=\sigma/\mu$, the probability that the ratio $Y/X$ (where Y is the bigger of the two values) is larger than $k$ is given by the following equation.

$$ p(k) = 2P(Y/X \ge k) = 2\Phi \left[ \frac{-\log(k)}{\sqrt{2\log(\theta^{2} + 1)}} \right], \quad k > 1$$

We want to derive a PDF that we can utilize for maximum likelihood estimation. Thus, I will substitute $k=e^{x} : x > 0$ and look at the proper CDF of the distribution.

$$ p(x \le X) = 2P(Y/X \ge e^{x}) = 1-2\Phi \left[ \frac{-x}{\sqrt{2\log(\theta^{2} + 1)}} \right], \quad x > 0$$

Differentiating with respect to $x$ gives:

Let $s = \sqrt{2\log(\theta^{2} + 1)}$, so that $p(x \le X) = 1 - 2\Phi\left(\dfrac{-x}{s}\right)$. Using $\dfrac{d}{dz}\Phi(z) = \phi(z) = \dfrac{1}{\sqrt{2\pi}}e^{-z^{2}/2}$ and the chain rule:

$$ p(x) = \frac{d}{dx}\left[1 - 2\Phi\left(\frac{-x}{s}\right)\right] = -2\phi\left(\frac{-x}{s}\right)\cdot\left(-\frac{1}{s}\right) = \frac{2}{s}\phi\left(\frac{-x}{s}\right) $$

Since $\phi$ is an even function, $\phi(-x/s) = \phi(x/s)$, so:

$$ p(x) = \frac{2}{s}\cdot\frac{1}{\sqrt{2\pi}}\exp\left(-\frac{x^{2}}{2s^{2}}\right) = \frac{2}{s\sqrt{2\pi}}\exp\left(-\frac{x^{2}}{2s^{2}}\right) $$

Substituting back $s^{2} = 2\log(\theta^{2}+1)$:

$$ p(x) = \frac{1}{\sqrt{\pi \log(\theta^{2}+1)}} \exp\left(-\frac{x^{2}}{4\log(\theta^{2}+1)}\right), \quad x > 0 $$

This PDF can be calculated in numpy via the following code.

In [ ]:
import numpy as np


def factor_pdf(fold_change, cv):
    """PDF of the fold change Y/X (larger over smaller) for two i.i.d. log-normal measurements with percent CV."""
    fold_change = np.asarray(fold_change)
    theta = cv / 100
    x = np.abs(np.log(fold_change))
    log_term = np.log(theta**2 + 1)
    return 1.0 / np.sqrt(np.pi * log_term) * np.exp(-(x**2) / (4 * log_term))


## Maximum Likelihood Estimation for theta

Since the PDF above is a half normal distribution with $\sigma^2 = 2\log(\theta^2 + 1)$, we can derive a maximum likelhood estimate for $\theta$. The general maximum likelihood estimate for $\sigma^2$ for a half normal is the following.

$$ \hat{\sigma}_{mle}^{2} = \frac{1}{N}\sum x_i^{2} $$ 

We have a 1-to-1 function from $\sigma^2$ to $\theta$, which is $\theta = f(\sigma^2) = \sqrt{\exp\left(\frac{1}{2}\sigma^2\right) - 1}$, so by the invariance of MLEs under transformation.

$$ \hat{\theta}_{mle} = \sqrt{\exp\left(\frac{1}{2}\hat{\sigma}_{mle}^2\right) - 1} = \sqrt{\exp\left(\frac{1}{2N}\sum_{i=1}^{N} x_i^{2}\right) - 1} $$

This can be calculated with the following numpy code.

In [ ]:
def factor_mle(fold_change):
    """Maximum likelihood estimate of the percent CV from an array of fold changes (Y/X, larger over smaller)."""
    fold_change = np.asarray(fold_change)
    x = np.log(fold_change)
    theta = np.sqrt(np.exp(np.mean(x**2) / 2) - 1)
    return theta * 100